In [1]:
# imports
import torch
import torch.nn as nn
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [2]:
# paths
CELEBRITY_CSV = "celebrity_dataset.csv"

MODEL_PATH = "mlp_finalExp_model.pth"

SCALER_PATH = "mlp_scaler_finalExp.pkl"

In [3]:
# loading MLP model
class MLPExperiment(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):

        return self.network(x)


model = MLPExperiment()

model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))

model.eval()

print("MLP Loaded")

MLP Loaded


In [4]:
# loading scaler
scaler = joblib.load(SCALER_PATH)

print("Scaler Loaded")

Scaler Loaded


In [5]:
# loading dataset
df = pd.read_csv(CELEBRITY_CSV)

print(df.shape)

df.head()

(200, 6)


,quality_score,best_similarity,margin,label,true_identity,gallery_identity
0,17.637470,0.550108,0.374149,1,Alice_Krige,Alice_Krige
1,17.637470,0.175959,0.005657,0,Alice_Krige,Joshua_Jackson
2,19.537975,0.351924,0.187498,1,Alyssa_Milano,Alyssa_Milano
3,19.537975,0.164426,0.022867,0,Alyssa_Milano,Gates_McFadden
4,22.501440,0.751467,0.618040,1,Amaury_Nolasco,Amaury_Nolasco


In [6]:
# MLP evaluation
X = df[
    [
        "quality_score",
        "best_similarity",
        "margin",
    ]
]

y_true = df["label"]

X_scaled = scaler.transform(X)

X_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32,
)

with torch.no_grad():

    outputs = model(X_tensor)

    probabilities = torch.softmax(
        outputs,
        dim=1,
    )

    y_pred = torch.argmax(
        outputs,
        dim=1,
    ).numpy()

In [7]:
# metrics
accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0,
)

cm = confusion_matrix(
    y_true,
    y_pred,
)

TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print("MLP RESULTS")
print("-------------------------")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

print()

print("TAR:", TAR)
print("FRR:", FRR)
print("TRR:", TRR)
print("FAR:", FAR)

MLP RESULTS
-------------------------
Accuracy : 0.775
Precision: 1.0
Recall   : 0.55
F1       : 0.7096774193548387

TAR: 0.55
FRR: 0.45
TRR: 1.0
FAR: 0.0


In [8]:
# threshold baseline
threshold_pred = (df["best_similarity"] >= 0.6).astype(int)

accuracy = accuracy_score(
    y_true,
    threshold_pred,
)

precision = precision_score(
    y_true,
    threshold_pred,
    zero_division=0,
)

recall = recall_score(
    y_true,
    threshold_pred,
    zero_division=0,
)

f1 = f1_score(
    y_true,
    threshold_pred,
    zero_division=0,
)

cm = confusion_matrix(
    y_true,
    threshold_pred,
)

TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print("THRESHOLD RESULTS")
print("-------------------------")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

print()

print("TAR:", TAR)
print("FRR:", FRR)
print("TRR:", TRR)
print("FAR:", FAR)

THRESHOLD RESULTS
-------------------------
Accuracy : 0.52
Precision: 1.0
Recall   : 0.04
F1       : 0.07692307692307693

TAR: 0.04
FRR: 0.96
TRR: 1.0
FAR: 0.0


In [9]:
# comparison table
comparison = pd.DataFrame(
    [
        {
            "Method": "Threshold",
            "Accuracy": accuracy_score(
                y_true,
                threshold_pred,
            ),
            "Precision": precision_score(
                y_true,
                threshold_pred,
                zero_division=0,
            ),
            "Recall": recall_score(
                y_true,
                threshold_pred,
                zero_division=0,
            ),
            "F1": f1_score(
                y_true,
                threshold_pred,
                zero_division=0,
            ),
        },
        {
            "Method": "Quality-aware MLP",
            "Accuracy": accuracy_score(
                y_true,
                y_pred,
            ),
            "Precision": precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "Recall": recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "F1": f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
        },
    ]
)

display(comparison.round(4))

,Method,Accuracy,Precision,Recall,F1
0,Threshold,0.520,1.0,0.04,0.0769
1,Quality-aware MLP,0.775,1.0,0.55,0.7097


In [10]:
# confusion matrices
print("MLP")

print(
    confusion_matrix(
        y_true,
        y_pred,
    )
)

print()

print("Threshold")

print(
    confusion_matrix(
        y_true,
        threshold_pred,
    )
)

MLP
[[100   0]
 [ 45  55]]

Threshold
[[100   0]
 [ 96   4]]


In [11]:
# dataset stats
print("Label Counts")

print(df["label"].value_counts())

print()

print("Overall Statistics")

print(
    df[
        [
            "quality_score",
            "best_similarity",
            "margin",
        ]
    ].describe()
)

print()

print("Positive Samples")

print(
    df[df["label"] == 1][
        [
            "quality_score",
            "best_similarity",
            "margin",
        ]
    ].describe()
)

print()

print("Negative Samples")

print(
    df[df["label"] == 0][
        [
            "quality_score",
            "best_similarity",
            "margin",
        ]
    ].describe()
)

print()

print("Predicted Accepts")

print(y_pred.sum())

print("Actual Accepts")

print(df["label"].sum())

Label Counts
label
1    100
0    100
Name: count, dtype: int64

Overall Statistics
       quality_score  best_similarity      margin
count     200.000000       200.000000  200.000000
mean       20.962221         0.279617    0.121734
std         2.457069         0.144405    0.134604
min        15.659685        -0.027753   -0.166733
25%        19.367360         0.165137    0.013649
50%        20.499352         0.215180    0.058100
75%        22.719759         0.383425    0.224874
max        26.945877         0.751467    0.618040

Positive Samples
       quality_score  best_similarity      margin
count     100.000000       100.000000  100.000000
mean       20.962221         0.389100    0.218967
std         2.463266         0.128716    0.129256
min        15.659685        -0.027753   -0.166733
25%        19.367360         0.299714    0.130492
50%        20.499352         0.383637    0.225920
75%        22.719759         0.471106    0.302683
max        26.945877         0.751467    0.618040